In [1]:
import pandas as pd
import numpy as np
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# 1. VERİ YÜKLEME VE DATA AUGMENTATION (Orijinal Veriyi Ekleme)
# ---------------------------------------------------------
print("--- 1. Veriler Yükleniyor ve Birleştiriliyor ---")
train_df = pd.read_csv("../data/raw/train.csv", index_col='id')
test_df = pd.read_csv("../data/raw/test.csv", index_col='id')

# Orijinal veriyi Kaggle forumunda paylaşılan IBM GitHub linkinden çekiyoruz
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
orig_df = pd.read_csv(url)

# Orijinal verinin index'i 'customerID' olarak geldiği için onu 'id' yapıp index'e atıyoruz
orig_df = orig_df.rename(columns={'customerID': 'id'}).set_index('id')

# Hedef değişkenleri ayırıyoruz (Metinleri 1-0 yapıyoruz)
y_train = train_df['Churn'].map({'Yes': 1, 'No': 0})
train_df.drop('Churn', axis=1, inplace=True)

y_orig = orig_df['Churn'].map({'Yes': 1, 'No': 0})
orig_df.drop('Churn', axis=1, inplace=True)

# Train ve Orijinal veriyi alt alta birleştirerek veri setimizi büyütüyoruz
X_train_full = pd.concat([train_df, orig_df])
y_train_full = pd.concat([y_train, y_orig])

print(f"✅ Orijinal veri eklendi! Yeni eğitim seti boyutu: {len(X_train_full)} satır.")

# ---------------------------------------------------------
# 2. FEATURE ENGINEERING (Öznitelik Mühendisliği)
# ---------------------------------------------------------
print("\n--- 2. Feature Engineering Uygulanıyor ---")
# Encoding sırasında kolon uyumsuzluğu olmaması için test setini de dahil edip işlem yapıyoruz
combined_df = pd.concat([X_train_full, test_df], keys=['train', 'test'])

# TotalCharges düzeltmesi
combined_df['TotalCharges'] = pd.to_numeric(combined_df['TotalCharges'], errors='coerce').fillna(0)

# Yeni Özelliklerimiz
combined_df['Avg_Monthly_Spend'] = combined_df['TotalCharges'] / (combined_df['tenure'] + 1)

# Kategorik Encoding
categorical_cols = combined_df.select_dtypes(include=['object']).columns
combined_df_encoded = pd.get_dummies(combined_df, columns=categorical_cols, drop_first=True)

# Combo Özellikler (Encoding sonrası kolon isimleriyle yapıyoruz)
if 'InternetService_Fiber optic' in combined_df_encoded.columns and 'PaymentMethod_Electronic check' in combined_df_encoded.columns:
    combined_df_encoded['High_Risk_Combo'] = ((combined_df_encoded['InternetService_Fiber optic'] == 1) & 
                                              (combined_df_encoded['PaymentMethod_Electronic check'] == 1)).astype(int)

if 'Contract_Two year' in combined_df_encoded.columns:
    combined_df_encoded['Loyalty_Score'] = combined_df_encoded['tenure'] * combined_df_encoded['Contract_Two year']

# Veriyi tekrar Train ve Test olarak ayırıyoruz
X_train_final = combined_df_encoded.xs('train')
X_test_final = combined_df_encoded.xs('test')

print(f"✅ İşlem tamam! Son eğitim seti boyutu: {X_train_final.shape}")

# ---------------------------------------------------------
# 3. OPTUNA İLE HİPERPARAMETRE OPTİMİZASYONU
# ---------------------------------------------------------
print("\n--- 3. Optuna Optimizasyonu Başlıyor (LightGBM) ---")

def objective(trial):
    # Optuna'nın deneyeceği parametre aralıkları
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 800),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
        'verbose': -1
    }
    
    # Optuna hızlı çalışsın diye 3-Fold yapıyoruz
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_auc = 0
    
    for train_idx, valid_idx in skf.split(X_train_final, y_train_full):
        X_tr, y_tr = X_train_final.iloc[train_idx], y_train_full.iloc[train_idx]
        X_va, y_va = X_train_final.iloc[valid_idx], y_train_full.iloc[valid_idx]
        
        model = LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_va)[:, 1]
        oof_auc += roc_auc_score(y_va, preds) / skf.n_splits
        
    return oof_auc

# Çalışma ortamı yaratıp Optuna'ya "AUC skorunu maksimize et" diyoruz
study = optuna.create_study(direction='maximize')
# Şimdilik zaman kazanmak için 10 farklı kombinasyon deneyecek. (Eğer vaktin varsa bunu 50 yapabilirsin!)
study.optimize(objective, n_trials=10)

print(f"\n🏆 Optuna'nın Bulduğu En İyi AUC Skoru: {study.best_value:.4f}")
print(f"✨ En İyi Parametreler: {study.best_params}")

C:\Users\Ayşe Paycı\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- 1. Veriler Yükleniyor ve Birleştiriliyor ---
✅ Orijinal veri eklendi! Yeni eğitim seti boyutu: 601237 satır.

--- 2. Feature Engineering Uygulanıyor ---


[I 2026-03-03 18:26:00,480] A new study created in memory with name: no-name-f64e800d-edbc-447a-b457-738b83995e0d


✅ İşlem tamam! Son eğitim seti boyutu: (601237, 33)

--- 3. Optuna Optimizasyonu Başlıyor (LightGBM) ---


[I 2026-03-03 18:26:13,164] Trial 0 finished with value: 0.9144564823888022 and parameters: {'n_estimators': 641, 'learning_rate': 0.019532906910035286, 'max_depth': 5, 'subsample': 0.5507692007237668, 'colsample_bytree': 0.8770750401565737}. Best is trial 0 with value: 0.9144564823888022.
[I 2026-03-03 18:26:17,988] Trial 1 finished with value: 0.9135219568621451 and parameters: {'n_estimators': 202, 'learning_rate': 0.030634846610858624, 'max_depth': 6, 'subsample': 0.6852771866171149, 'colsample_bytree': 0.8220324618370574}. Best is trial 0 with value: 0.9144564823888022.
[I 2026-03-03 18:26:25,814] Trial 2 finished with value: 0.9121422833251581 and parameters: {'n_estimators': 603, 'learning_rate': 0.012780309973661731, 'max_depth': 3, 'subsample': 0.8608326741398125, 'colsample_bytree': 0.8849618929241158}. Best is trial 0 with value: 0.9144564823888022.
[I 2026-03-03 18:26:39,451] Trial 3 finished with value: 0.9144622518733423 and parameters: {'n_estimators': 768, 'learning_rat


🏆 Optuna'nın Bulduğu En İyi AUC Skoru: 0.9151
✨ En İyi Parametreler: {'n_estimators': 369, 'learning_rate': 0.05463091884804394, 'max_depth': 7, 'subsample': 0.8796307578392997, 'colsample_bytree': 0.9158469772433917}


In [2]:
print("\n--- 4. Nihai Model Eğitimi (Optuna Parametreleriyle) ---")

# Optuna'nın bulduğu en iyi parametreleri alıyoruz ve birkaç zorunlu ayar ekliyoruz
best_params = study.best_params
best_params['random_state'] = 42
best_params['verbose'] = -1

# Test tahminleri için boş dizi
test_preds_final = np.zeros(len(X_test_final))

# Daha stabil bir test tahmini için 5-Fold kullanıyoruz
skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_auc_final = np.zeros(len(X_train_final))

for fold, (train_idx, valid_idx) in enumerate(skf_final.split(X_train_final, y_train_full)):
    X_tr, y_tr = X_train_final.iloc[train_idx], y_train_full.iloc[train_idx]
    X_va, y_va = X_train_final.iloc[valid_idx], y_train_full.iloc[valid_idx]
    
    # Modeli Optuna parametreleriyle kuruyoruz
    model_final = LGBMClassifier(**best_params)
    model_final.fit(X_tr, y_tr)
    
    valid_preds = model_final.predict_proba(X_va)[:, 1]
    oof_auc_final[valid_idx] = valid_preds
    
    # Test setini her fold'da tahmin edip ortalamasını alıyoruz
    test_preds_final += model_final.predict_proba(X_test_final)[:, 1] / skf_final.n_splits
    
    print(f"Final Model Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

print(f"\n🚀 Orijinal Veri + Optuna Ortalama OOF Skoru: {roc_auc_score(y_train_full, oof_auc_final):.4f}")

# Kaggle Gönderim Dosyası
submission_optuna = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds_final})
submission_optuna.to_csv("../submissions/submission_lgbm_optuna_augmented.csv", index=False)
print("✅ Mükemmel gönderim dosyası 'submissions' klasörüne kaydedildi!")


--- 4. Nihai Model Eğitimi (Optuna Parametreleriyle) ---
Final Model Fold 1 AUC: 0.9155
Final Model Fold 2 AUC: 0.9160
Final Model Fold 3 AUC: 0.9137
Final Model Fold 4 AUC: 0.9150
Final Model Fold 5 AUC: 0.9159

🚀 Orijinal Veri + Optuna Ortalama OOF Skoru: 0.9152
✅ Mükemmel gönderim dosyası 'submissions' klasörüne kaydedildi!


In [4]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print("\n--- 5. GRANDMASTER TAKTİĞİ: Pseudo Labeling (Yalancı Etiketleme) ---")

# 1. En başarılı bulduğumuz gönderim dosyamızı (Optuna) okuyoruz
# DÜZELTME: index_col='id' ekleyerek test verimizle hizaladık!
best_submission = pd.read_csv("../submissions/submission_lgbm_optuna_augmented.csv", index_col='id')

# 2. Modelin "ÇOK EMİN" olduğu test verilerini seçiyoruz (Örn: İhtimal > %90 veya < %10)
high_confidence_mask = (best_submission['Churn'] > 0.90) | (best_submission['Churn'] < 0.10)

# Artık indexler aynı olduğu için filtreleme kusursuz çalışacak
pseudo_X = X_test_final[high_confidence_mask].copy()

# Olasılıkları 1 ve 0'a (Kesin Churn = 1, Kesin Kalıcı = 0) dönüştürüp Y (hedef) yapıyoruz
pseudo_y_values = np.where(best_submission.loc[high_confidence_mask, 'Churn'] > 0.90, 1, 0)
pseudo_y = pd.Series(pseudo_y_values, index=pseudo_X.index, name='Churn')

# 3. YALANCI ETİKETLERİ EĞİTİM SETİNE EKLİYORUZ
X_train_pseudo = pd.concat([X_train_final, pseudo_X])
y_train_pseudo = pd.concat([y_train_full, pseudo_y])

print(f"Orijinal Eğitim Boyutu: {len(X_train_final)}")
print(f"Modelin Çok Emin Olup Testten Kopardığı Satır Sayısı: {len(pseudo_X)}")
print(f"🚀 YEPYENİ DEVASA EĞİTİM BOYUTU: {len(X_train_pseudo)}")

# 4. YENİ VERİYLE XGBOOST EĞİTİMİ (5-Fold)
print("\nModel Pseudo-Label verisiyle baştan eğitiliyor...")

skf_pseudo = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_auc_pseudo = np.zeros(len(X_train_pseudo))
test_preds_pseudo = np.zeros(len(X_test_final))

for fold, (train_idx, valid_idx) in enumerate(skf_pseudo.split(X_train_pseudo, y_train_pseudo)):
    X_tr, y_tr = X_train_pseudo.iloc[train_idx], y_train_pseudo.iloc[train_idx]
    X_va, y_va = X_train_pseudo.iloc[valid_idx], y_train_pseudo.iloc[valid_idx]
    
    # Güçlü bir XGBoost kuruyoruz
    model_xgb_pl = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="auc"
    )
    
    model_xgb_pl.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    
    valid_preds = model_xgb_pl.predict_proba(X_va)[:, 1]
    oof_auc_pseudo[valid_idx] = valid_preds
    
    # Gerçek test setini tahmin ediyoruz
    test_preds_pseudo += model_xgb_pl.predict_proba(X_test_final)[:, 1] / skf_pseudo.n_splits
    
    print(f"Pseudo Fold {fold+1} AUC: {roc_auc_score(y_va, valid_preds):.4f}")

print(f"\n🏆 Pseudo Labeling (OOF) AUC Skoru: {roc_auc_score(y_train_pseudo, oof_auc_pseudo):.4f}")

# 5. Gönderim Dosyası Hazırlama
submission_pl = pd.DataFrame({'id': X_test_final.index, 'Churn': test_preds_pseudo})
submission_pl.to_csv("../submissions/submission_pseudo_labeling.csv", index=False)
print("✅ Yalancı Etiketleme (PL) gönderim dosyası 'submissions' klasörüne kaydedildi!")


--- 5. GRANDMASTER TAKTİĞİ: Pseudo Labeling (Yalancı Etiketleme) ---
Orijinal Eğitim Boyutu: 601237
Modelin Çok Emin Olup Testten Kopardığı Satır Sayısı: 142687
🚀 YEPYENİ DEVASA EĞİTİM BOYUTU: 743924

Model Pseudo-Label verisiyle baştan eğitiliyor...
Pseudo Fold 1 AUC: 0.9351
Pseudo Fold 2 AUC: 0.9345
Pseudo Fold 3 AUC: 0.9351
Pseudo Fold 4 AUC: 0.9343
Pseudo Fold 5 AUC: 0.9342

🏆 Pseudo Labeling (OOF) AUC Skoru: 0.9346
✅ Yalancı Etiketleme (PL) gönderim dosyası 'submissions' klasörüne kaydedildi!
